In [29]:
import requests
import concurrent.futures
import json
import os
import sys
from typing import Dict, Optional
from datetime import datetime, timedelta
from dotenv import load_dotenv

In [ ]:
# Load environment variables
load_dotenv()

# ============================================
# CONFIGURATION
# ============================================
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY", "")
GEMINI_MODEL = "gemini-3.1-flash-lite"
GEMINI_TEMPERATURE = 0.7

DEFAULT_LOCATION = {
    'lat': 19.0760, 'lon': 72.8777,
    'city': 'Mumbai', 'state': 'Maharashtra', 'country': 'India'
    }

In [31]:
WEATHER_API_URL = "https://api.open-meteo.com/v1/forecast"
WEATHER_PARAMS = "temperature_2m,relative_humidity_2m,wind_speed_10m,pressure_msl,uv_index"
OUTPUT_FILE = "h2o_ai_report_H2O.txt"

In [32]:
# ============================================
# LOCATION SERVICE
# ============================================
class LocationService:
    """Gets user location using multiple IP geolocation APIs"""
    
    def __init__(self):
        self.cached_location = None
    
    def get_location(self) -> Dict:
        if self.cached_location:
            return self.cached_location
        
        location = self._get_consensus_location()
        if location:
            self.cached_location = location
            return location
        return DEFAULT_LOCATION
    
    def _get_consensus_location(self) -> Optional[Dict]:
        apis = [self._try_ipapi, self._try_ipinfo, self._try_ipapi_co]
        try:
            import geocoder
            apis.append(self._try_geocoder)
        except ImportError:
            pass
        
        results = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            futures = [executor.submit(api) for api in apis]
            for future in concurrent.futures.as_completed(futures):
                try:
                    result = future.result()
                    if result and result.get('city') != 'Unknown':
                        results.append(result)
                except:
                    pass
        
        if not results:
            return None
        
        city_votes = {}
        for r in results:
            city = r['city']
            city_votes[city] = city_votes.get(city, 0) + 1
        
        best_city = max(city_votes, key=city_votes.get)
        for r in results:
            if r['city'] == best_city:
                r['confidence'] = f"{city_votes[best_city]}/{len(results)} APIs agreed"
                return r
        return results[0]
    
    def _try_ipapi(self):
        try:
            r = requests.get('http://ip-api.com/json/', timeout=5)
            d = r.json()
            if d.get('status') == 'success':
                return {'lat': d['lat'], 'lon': d['lon'], 'city': d.get('city',''), 'state': d.get('regionName',''), 'country': d.get('country',''), 'source': 'ip-api'}
        except: pass
    
    def _try_ipinfo(self):
        try:
            r = requests.get('https://ipinfo.io/json', timeout=5)
            d = r.json()
            if 'loc' in d:
                lat, lon = d['loc'].split(',')
                return {'lat': float(lat), 'lon': float(lon), 'city': d.get('city',''), 'state': d.get('region',''), 'country': d.get('country',''), 'source': 'ipinfo'}
        except: pass
    
    def _try_geocoder(self):
        try:
            import geocoder
            g = geocoder.ip('me')
            if g.ok and g.city:
                return {'lat': g.latlng[0] if g.latlng else 0, 'lon': g.latlng[1] if g.latlng else 0, 'city': g.city, 'state': g.state or '', 'country': g.country or '', 'source': 'geocoder'}
        except: pass
    
    def _try_ipapi_co(self):
        try:
            r = requests.get('https://ipapi.co/json/', timeout=5)
            d = r.json()
            if 'error' not in d and d.get('city'):
                return {'lat': d.get('latitude'), 'lon': d.get('longitude'), 'city': d.get('city',''), 'state': d.get('region',''), 'country': d.get('country_name',''), 'source': 'ipapi.co'}
        except: pass

In [33]:
# ============================================
# WEATHER SERVICE
# ============================================
class WeatherService:
    """Fetches weather data from Open-Meteo API"""
    
    def get_weather(self, lat, lon):
        params = {
            "latitude": lat,
            "longitude": lon,
            "current": WEATHER_PARAMS,
            "timezone": "auto"
        }
        try:
            r = requests.get(WEATHER_API_URL, params=params, timeout=10)
            r.raise_for_status()
            data = r.json()
            current = data.get("current", {})
            return {
                "temperature": current.get("temperature_2m"),
                "humidity": current.get("relative_humidity_2m"),
                "wind_speed": current.get("wind_speed_10m"),
                "pressure": current.get("pressure_msl"),
                "uv_index": current.get("uv_index")
            }
        except Exception as e:
            print(f"❌ Weather error: {e}")
            return None

In [34]:
# ============================================
# GEMINI AI AGENT
# ============================================
class GeminiAgent:
    """AI Agent using Google Gemini for intelligent recommendations"""
    
    def __init__(self):
        try:
            from langchain_google_genai import ChatGoogleGenerativeAI
            self.llm = ChatGoogleGenerativeAI(
                google_api_key=GEMINI_API_KEY,
                model=GEMINI_MODEL,
                temperature=GEMINI_TEMPERATURE
            )
            self.use_langchain = True
        except ImportError:
            print("⚠️  LangChain not installed, using direct Gemini API")
            self.use_langchain = False
            self.api_key = GEMINI_API_KEY
            self.api_url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent?key={GEMINI_API_KEY}"
    
    def _call_gemini(self, prompt: str) -> str:
        """Call Gemini API - uses LangChain if available, else direct API"""
        if self.use_langchain:
            from langchain.schema import HumanMessage
            try:
                response = self.llm.invoke([HumanMessage(content=prompt)])
                return response.content
            except Exception as e:
                print(f"⚠️  LangChain Gemini error: {e}")
                return None
        
        # Direct API call fallback
        try:
            headers = {'Content-Type': 'application/json'}
            data = {
                "contents": [{"parts": [{"text": prompt}]}],
                "generationConfig": {
                    "temperature": GEMINI_TEMPERATURE,
                    "maxOutputTokens": 500
                }
            }
            response = requests.post(self.api_url, headers=headers, json=data, timeout=30)
            response.raise_for_status()
            result = response.json()
            return result['candidates'][0]['content']['parts'][0]['text']
        except Exception as e:
            print(f"⚠️  Direct Gemini API error: {e}")
            return None
    
    def generate_hydration_advice(self, location: dict, weather: dict) -> str:
        prompt = f"""You are H2O AI, an expert hydration advisor. Provide personalized hydration advice.

LOCATION: {location.get('city', 'Unknown')}, {location.get('state', '')}, {location.get('country', '')}

CURRENT WEATHER:
- Temperature: {weather['temperature']}°C
- Humidity: {weather['humidity']}%
- Wind Speed: {weather['wind_speed']} km/h
- UV Index: {weather['uv_index']}

Provide:
1. Today's hydration risk level (low/moderate/high/extreme)
2. Recommended water intake in litres
3. Best times to drink water today
4. Special considerations for this weather

Be conversational, friendly, and specific. Keep under 150 words."""
        
        response = self._call_gemini(prompt)
        return response if response else self._fallback_hydration(weather)
    
    def generate_food_recommendations(self, location: dict, weather: dict) -> str:
        prompt = f"""You are a nutrition advisor. Recommend foods for today's weather.

Location: {location.get('city', 'your area')}, {location.get('country', '')}
Weather: {weather['temperature']}°C, Humidity: {weather['humidity']}%

Suggest specific foods that:
- Support hydration
- Match the weather conditions
- Include local cuisine options
- Explain why they're good for today

Keep under 100 words. Be practical and specific."""
        
        response = self._call_gemini(prompt)
        return response if response else self._fallback_food(weather)
    
    def generate_activity_suggestions(self, weather: dict) -> str:
        prompt = f"""You are a fitness advisor. Suggest activities for today's weather.

Weather: {weather['temperature']}°C, Humidity: {weather['humidity']}%, UV Index: {weather['uv_index']}

Recommend:
- Specific activities (not generic)
- Best time of day for activities
- Safety precautions
- Hydration tips during activity

Keep under 80 words."""
        
        response = self._call_gemini(prompt)
        return response if response else self._fallback_activity(weather)
    
    def generate_daily_summary(self, location: dict, weather: dict) -> str:
        prompt = f"""You are H2O AI, a friendly health assistant. Create a daily summary.

Location: {location.get('city', 'your city')}, {location.get('country', '')}
Weather: {weather['temperature']}°C, Humidity: {weather['humidity']}%, UV: {weather['uv_index']}

Include: Weather overview, hydration goal, one food tip, one activity tip, and encouragement.
Be warm and motivating. Under 120 words."""
        
        response = self._call_gemini(prompt)
        if response:
            return response
        return f"Today in {location.get('city', 'your area')}: {weather['temperature']}°C with {weather['humidity']}% humidity. Stay hydrated with at least 2.5-3L of water. Include water-rich foods like fruits. Great day for some outdoor activity!"
    
    def calculate_water_intake(self, weather: dict) -> dict:
        prompt = f"""Calculate recommended daily water intake based on weather.

Temperature: {weather['temperature']}°C
Humidity: {weather['humidity']}%

Rules:
- Base: 2.5L for average adult
- Hot (>30°C): +0.5 to 1.0L
- High humidity (>70%): +0.2 to 0.5L
- Cold (<10°C): -0.2L
- Dry (<40% humidity): +0.2L

Return ONLY valid JSON (no markdown, no explanation):
{{"water_litres": 3.1, "interval_minutes": 20, "serving_ml": 180, "risk_level": "high"}}

risk_level must be: low, moderate, high, or extreme"""
        
        response = self._call_gemini(prompt)
        
        if response:
            try:
                content = response.strip()
                if content.startswith("```"):
                    content = content.split("\n", 1)[1]
                    if content.endswith("```"):
                        content = content.rsplit("\n", 1)[0]
                
                start = content.find('{')
                end = content.rfind('}') + 1
                if start >= 0 and end > start:
                    data = json.loads(content[start:end])
                    required = ['water_litres', 'interval_minutes', 'serving_ml', 'risk_level']
                    if all(k in data for k in required):
                        return data
            except:
                pass
        
        return self._fallback_water_calculation(weather)
    
    def _fallback_water_calculation(self, weather: dict) -> dict:
        temp = weather['temperature']
        humidity = weather['humidity']
        
        water = 2.5
        if temp >= 35: water += 1.0; risk = "extreme"; interval = 15; serving = 150
        elif temp >= 30: water += 0.6; risk = "high"; interval = 20; serving = 180
        elif temp >= 20: water += 0.3; risk = "moderate"; interval = 30; serving = 200
        elif temp <= 10: water -= 0.2; risk = "low"; interval = 60; serving = 200
        else: risk = "low"; interval = 45; serving = 200
        
        if humidity >= 70: water += 0.3
        elif humidity < 40: water -= 0.1
        
        return {
            "water_litres": round(water, 1),
            "interval_minutes": interval,
            "serving_ml": serving,
            "risk_level": risk
        }
    
    def _fallback_hydration(self, weather: dict) -> str:
        temp = weather['temperature']
        humidity = weather['humidity']
        if temp > 30:
            return f"⚠️ High temperature ({temp}°C) with {humidity}% humidity increases dehydration risk. Drink 3-3.5L today. Sip water every 15-20 minutes. Keep a water bottle with you always."
        elif temp > 20:
            return f"✅ Moderate weather at {temp}°C. Aim for 2.5-3L of water. Drink consistently throughout the day. Your body needs regular hydration even in comfortable weather."
        else:
            return f"💡 Cool weather ({temp}°C) can mask thirst. Maintain 2-2.5L daily. Warm water or herbal tea counts toward your intake."
    
    def _fallback_food(self, weather: dict) -> str:
        temp = weather['temperature']
        if temp > 30:
            return "Water-rich foods: watermelon, cucumber, oranges, coconut water. Light salads and smoothies. Avoid heavy, oily, spicy foods."
        elif temp > 20:
            return "Fresh seasonal vegetables and fruits. Light proteins like yogurt, paneer. Stay away from excessive fried food."
        else:
            return "Warm meals: soups, stews, root vegetables. Herbal teas with ginger or tulsi. Nuts and seeds for warmth."
    
    def _fallback_activity(self, weather: dict) -> str:
        temp = weather['temperature']
        uv = weather['uv_index']
        if temp > 35:
            return "🏊‍♂️ Indoor activities or swimming. Avoid outdoors 11 AM-4 PM. Wear sun protection, carry water."
        elif temp > 30:
            return "🚴‍♂️ Early morning/evening walks, jogging, cycling. Stay in shade, hydrate frequently."
        elif temp > 20:
            return "🏃‍♂️ Perfect for outdoor activities! Running, hiking, sports. Carry water for longer sessions."
        else:
            return "🧘‍♀️ Brisk walking, yoga. Layer up if going outside. Warm up properly before exercise."

In [35]:
# ============================================
# HYDRATION SCHEDULER
# ============================================
class HydrationScheduler:
    """Creates intelligent hydration schedules"""
    
    def generate_schedule(self, water_data: dict, wake_up: int = 8, sleep: int = 22):
        total_ml = water_data['water_litres'] * 1000
        interval = water_data['interval_minutes']
        serving = water_data['serving_ml']
        
        waking_minutes = (sleep - wake_up) * 60
        max_servings = waking_minutes // interval
        servings = min(int(total_ml / serving), max_servings)
        
        schedule = []
        current_time = datetime.strptime(f"{wake_up}:00", "%H:%M")
        
        for i in range(servings):
            schedule.append({
                'time': current_time.strftime("%I:%M %p"),
                'amount_ml': serving,
                'percentage': round(((i+1)*serving / total_ml) * 100)
            })
            current_time += timedelta(minutes=interval)
        
        return {
            'total_ml': int(total_ml),
            'serving_ml': serving,
            'interval_minutes': interval,
            'total_servings': servings,
            'schedule': schedule,
            'time_range': f"{wake_up}:00 - {sleep}:00"
        }

In [36]:
# ============================================
# MAIN APPLICATION
# ============================================
class H2OAIAssistant:
    """Main AI-powered hydration assistant"""
    
    def __init__(self):
        self.location_service = LocationService()
        self.weather_service = WeatherService()
        self.ai_agent = GeminiAgent()
        self.scheduler = HydrationScheduler()
    
    def run(self):
        lines = []
        
        # Header
        lines.append("=" * 70)
        lines.append("   💧  H2O AI - Context-Aware Hydration Intelligence  💧")
        lines.append("   Powered by Google Gemini AI")
        lines.append("=" * 70)
        
        # Phase 1: Context
        lines.append("\n🧠 PHASE 1: CONTEXT GATHERING")
        lines.append("-" * 50)
        
        lines.append("   🔍 Detecting location...")
        location = self.location_service.get_location()
        lines.append(f"   📍 {location['city']}, {location.get('state', '')}, {location['country']}")
        
        lines.append("   🌤️  Fetching weather...")
        weather = self.weather_service.get_weather(location['lat'], location['lon'])
        
        if not weather:
            lines.append("   ❌ Weather unavailable")
            self._output(lines)
            return
        
        lines.append(f"   🌡️  {weather['temperature']}°C | 💧 {weather['humidity']}% | ☀️ UV: {weather['uv_index']}")
        lines.append("   ✅ Context gathered")
        
        # Phase 2: AI Analysis
        lines.append("\n🤖 PHASE 2: GEMINI AI ANALYSIS")
        lines.append("-" * 50)
        lines.append("   🧠 Analyzing with Gemini...")
        
        water_data = self.ai_agent.calculate_water_intake(weather)
        lines.append(f"   💧 {water_data['water_litres']}L/day | Risk: {water_data['risk_level'].upper()}")
        
        # Phase 3: AI Recommendations
        lines.append("\n📋 PHASE 3: AI RECOMMENDATIONS")
        lines.append("=" * 70)
        
        # Summary
        lines.append("\n📝 DAILY HEALTH SUMMARY")
        lines.append("-" * 50)
        summary = self.ai_agent.generate_daily_summary(location, weather)
        for line in summary.split('\n'):
            if line.strip():
                lines.append(f"   {line.strip()}")
        
        # Hydration
        lines.append("\n💧 HYDRATION ADVICE")
        lines.append("-" * 50)
        hydration = self.ai_agent.generate_hydration_advice(location, weather)
        for line in hydration.split('\n'):
            if line.strip():
                lines.append(f"   {line.strip()}")
        
        # Food
        lines.append("\n🍽️  FOOD RECOMMENDATIONS")
        lines.append("-" * 50)
        food = self.ai_agent.generate_food_recommendations(location, weather)
        for line in food.split('\n'):
            if line.strip():
                lines.append(f"   {line.strip()}")
        
        # Activity
        lines.append("\n🏃 ACTIVITY SUGGESTIONS")
        lines.append("-" * 50)
        activity = self.ai_agent.generate_activity_suggestions(weather)
        for line in activity.split('\n'):
            if line.strip():
                lines.append(f"   {line.strip()}")
        
        # Phase 4: Schedule
        lines.append("\n⏰ PHASE 4: HYDRATION SCHEDULE")
        lines.append("-" * 50)
        
        schedule_data = self.scheduler.generate_schedule(water_data)
        
        lines.append(f"   📊 Total: {schedule_data['total_ml']}ml")
        lines.append(f"   🥤 Per serving: {schedule_data['serving_ml']}ml")
        lines.append(f"   ⏱️  Every: {schedule_data['interval_minutes']} min")
        lines.append(f"   🕐 Hours: {schedule_data['time_range']}")
        lines.append("")
        
        for entry in schedule_data['schedule']:
            bar = "▓" * (entry['percentage'] // 10) + "░" * (10 - entry['percentage'] // 10)
            lines.append(f"   🥤 {entry['time']} → {entry['amount_ml']}ml  [{bar}] {entry['percentage']}%")
        
        # Footer
        lines.append("\n" + "=" * 70)
        lines.append(f"   ✅ Generated at {datetime.now().strftime('%I:%M %p')}")
        lines.append(f"   📍 {location['city']}, {location['country']}")
        lines.append("   💙 Stay healthy, stay hydrated!")
        lines.append("=" * 70)
        
        self._output(lines)
    
    def _output(self, lines):
        """Save and print output"""
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            for line in lines:
                f.write(line + '\n')
        
        for line in lines:
            print(line)
        
        print(f"\n📄 Full report: {os.path.abspath(OUTPUT_FILE)}\n")

In [37]:
# ============================================
# ENTRY POINT
# ============================================
def main():
    print("\n" + "=" * 70)
    print("   💧  H2O AI - Context-Aware Hydration Assistant  💧")
    print("=" * 70)
    
    # Check API key
    if not GEMINI_API_KEY:
        print("\n❌ Google Gemini API Key not found!")
        print("\n   Quick Setup:")
        print("   1. Go to: https://aistudio.google.com/app/apikey")
        print("   2. Click 'Create API Key'")
        print("   3. Copy your key")
        print("   4. Create .env file: GEMINI_API_KEY=your_key_here")
        print("\n   Or set it directly in code (line 17)\n")
        return
    
    print("\n✅ Gemini API key found")
    print("🚀 Starting H2O AI...\n")
    
    try:
        assistant = H2OAIAssistant()
        assistant.run()
    except KeyboardInterrupt:
        print("\n\n👋 Goodbye! Stay hydrated!\n")
    except Exception as e:
        print(f"\n❌ Error: {e}\n")


if __name__ == "__main__":
    main()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



   💧  H2O AI - Context-Aware Hydration Assistant  💧

✅ Gemini API key found
🚀 Starting H2O AI...


❌ Error: No module named 'langchain.schema'

